# Module 04: Linear Regression

In [ ]:
# packages
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)

# set seed
seed = 4721

### We'll use the _Bikeshare_ data from ISLP for this activity. The metadata for _Bikeshare_ can be found [here](https://intro-stat-learning.github.io/ISLP/datasets/Bikeshare.html).

In [ ]:
# Load the data
Bikeshare = load_data('Bikeshare')

In [ ]:
# List the columns and their types
Bikeshare.dtypes

### We will predict the `bikers` column, which is the total of casual and registered bikers. 
Then we can create a matrix of potential independent variables (X) and a dependent variable vector (y).

In [ ]:
indep_vars = ['season', 'mnth', 'day', 'hr', 
              'holiday', 'weekday', 'workingday', 
              'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

X = Bikeshare[indep_vars]
y = Bikeshare['bikers']

### Before doing any other analyses, let's create training and test sets.

In [ ]:
X_train, X_test, y_train, y_test, Train, Test = train_test_split(X, y, Bikeshare, 
                                                                 random_state = seed, 
                                                                 test_size = 0.25, 
                                                                 shuffle = True)
                             

### We can first summarize the variables in the training data.

In [ ]:
Train.describe()

### Then we can create plots for each potential independent variable with `bikers` on training data.

In [ ]:
# Initialize the plots before drawing them
nrows = 4
ncols = 3
figsize = (5*nrows, 5*ncols)

fig, axes = subplots(nrows=nrows,
                     ncols=ncols,
                     figsize=figsize)

# Assign a grid location to each index
def range_to_grid(i, nrows, ncols):
    x=[]
    y=[]
    for n in range(nrows*ncols):
        x.append(n // ncols)
        y.append(n % ncols)
        # print(n,x[n],y[n]) # for testing this function
    return x[i],y[i]

# Plot the variables
for j in range(len(X_train.columns)):
    # print(range_to_grid(j,nrows,ncols)[0], range_to_grid(j,nrows,ncols)[1]) # testing
    axes[range_to_grid(j,nrows,ncols)[0],
         range_to_grid(j,nrows,ncols)[1]].plot(X_train.iloc[:,j], y_train, 'o')
    axes[range_to_grid(j,nrows,ncols)[0],
         range_to_grid(j,nrows,ncols)[1]].set_xlabel(X_train.columns[j])

### Explain why you can deduce that there likely are no missing values in this dataset.

In the Train.describe(), all the variables have the same count (6483) which means that the number of observations in the training set are the same. In case some values were missing, their columns' counts would have been lower than others. Moreover, the plots don't show any gaps or placeholders (Such as -999 or 0 that sometimes represent a missing value). Therefore, it is possible to conclude that there are no missing values in the data set.

### The variables that have approximately linear relationship with `bikers` are `temp`, `atemp`, `windspeed`, `workingday`, and `holiday`. 

It appears that time-related variables (`season`, `day`, `hr`, etc.) have non-linear relationships with `bikers`, so we'll hold off on using these variables until the next module.

In [ ]:
# correlation of potential linear variables
Train[['bikers', 'temp', 'atemp', 'windspeed', 'workingday', 'holiday']].corr()

### Explain why `temp` or `atemp` should be used to predict `bikers`, but NOT both.

In the given case, temperature and normalized temperature have a high correlation of approximately 0.99. Therefore, it is possible to state that both variables provide similar information to the model, and the inclusion of both is not necessary, as it introduces multicollinearity into the model. It will lead to the coefficients’ standard errors being overestimated, and their statistical significance being reduced. On the other hand, the addition of the second predictor will not bring much improvement to the model, as it will not increase the coefficient of determination by a significant margin, given that the two variables have a similar correlation with the bikers. Thus, the inclusion of the second predictor is not worth pursuing.

## First Linear Regression Model

### We can create a column to represent the intercept in our model. 

Some packages do this automatically, but we will leverage the ISLP version, which does not.

In [ ]:
X_train['intercept'] = np.ones(X_train.shape[0])
X_test['intercept'] = np.ones(X_test.shape[0])

### Now we'll build a simple linear regression model that uses only `temp` and an intercept to predict `bikers`.

In [ ]:
model_1 = sm.OLS(y_train, X_train[['intercept','temp']])
results_1 = model_1.fit()
summarize(results_1)

### Use this model to predict the number of bikers on a 68F day. 

_Hint_: Use the [metadata](https://intro-stat-learning.github.io/ISLP/datasets/Bikeshare.html) and the fact that 68F = 20C.

According to the metadata, temp is normalized: temperature in Celsius divided by 41. So 68°F = 20°C gives temp = 20/41 ≈ 0.488.

Using model 1: bikers = −8.311 + 309.437 × 0.488 ≈ 142.6, so about 143 bikers per hour.

### We will evaluate this model *three ways*: Using R^2, using MSE, and then visually. 

For the last two, we will compare on _Train_ and on _Test_.

In [ ]:
# R^2 on training
results_1.rsquared

In [ ]:
# Create helper functions for computing the mean squared error

def predict(X, model):
    # the built-in get_prediction tool returns an array, so we need to convert to a dataframe
    predictions_df = pd.DataFrame(model.get_prediction(X).predicted, columns=['y_hat'], index=X.index)
    return predictions_df['y_hat']

def mse(y, y_hat):
    # calculate the residual error for each individual record
    resid = y - y_hat
    # square the residual (hence "squared error")
    sq_resid = resid**2
    # calculate the sum of squared errors
    SSR = sum(sq_resid)
    # divide by the number of records to get the mean squared error
    MSE = SSR / y.shape[0]
    return MSE

In [ ]:
predictions_train_1 = predict(X_train[['intercept', 'temp']], results_1)
print('MSE train: ', mse(y_train, predictions_train_1))

predictions_test_1 = predict(X_test[['intercept', 'temp']], results_1)
print('MSE test: ', mse(y_test, predictions_test_1))

In [ ]:
# Define a function to draw a line given coefficients [credit to Hastie & Tibshirani]

def abline(ax, b, m, *args, **kwargs):
    "Add a line with slope m and intercept b to ax"
    xlim = ax.get_xlim()
    ylim = [m * xlim[0] + b, m * xlim[1] + b]
    ax.plot(xlim, ylim, *args, **kwargs)

In [ ]:
ax = Train.plot.scatter('temp', 'bikers')
ax.set_title("Plot of temp vs bikers (train)")
abline(ax,
       results_1.params[0],
       results_1.params[1],
       'r--',
       linewidth=3)

In [ ]:
ax = Test.plot.scatter('temp', 'bikers')
ax.set_title("Plot of temp vs bikers (test)")
abline(ax,
       results_1.params[0],
       results_1.params[1],
       'm--',
       linewidth=3)

### Using the same training set, build a new model that uses both temp and holiday to predict bikers.

In [ ]:
model_2 = sm.OLS(y_train, X_train[['intercept', 'temp', 'holiday']])
results_2 = model_2.fit()
summarize(results_2)

### Based on the summary of the new model, does it make sense to include `holiday` as a predictor? Why or why not?

No, holiday should not be included. In the summary, we can see that its coefficient is −9.60 with a standard error of 8.92, which gives a t-statistic of −1.08 and a p-value of 0.282 (greater than 0.05). This means that we cannot say there is an association between this variable and the bikers’ demand. Moreover, the 95% confidence interval (approximately -27 to +8) for the coefficient also includes 0, which means that it could even be positive. The change in temp coefficient (309.44 to 309.25) is also not significant enough for us to consider adding this variable.

### Compute the R^2 coefficient for the new model.

In [ ]:
results_2.rsquared

### Compute the MSE on the training and test sets for the new model. 

In [ ]:
predictions_train_2 = predict(X_train[['intercept', 'temp', 'holiday']], results_2)
print('MSE train: ', mse(y_train, predictions_train_2))

predictions_test_2 = predict(X_test[['intercept', 'temp', 'holiday']], results_2)
print('MSE test: ', mse(y_test, predictions_test_2))

### Based on the MSE calculations, is this model better or worse than the model containing only `temp`? Why or why not?

The model that includes both temp and holiday is technically somewhat better, but not by much. The training MSE decreased only slightly from 14223.58 to 14221.04, and the test MSE decreased only slightly from 14355.47 to 14349.57 (a decrease of less than 0.05%). One would generally expect that adding any variable to a linear regression would decrease the training MSE, even if the variable had no real predictive power. The test MSE was almost identical, and the coefficient for holiday was not statistically significant (p = 0.282). Overall, the two models were practically the same, and the simpler model that only included temp was better.

### Now build a model using `windspeed` instead of `holiday` as a predictor. 

In [ ]:
model_3 = sm.OLS(y_train, X_train[['intercept', 'temp', 'windspeed']])
results_3 = model_3.fit()
summarize(results_3)

### Compute the R^2 for this model.

In [ ]:
results_3.rsquared

### Compute the MSE for this model on train and test.

In [ ]:
predictions_train_3 = predict(X_train[['intercept', 'temp', 'windspeed']], results_3)
print('MSE train: ', mse(y_train, predictions_train_3))

predictions_test_3 = predict(X_test[['intercept', 'temp', 'windspeed']], results_3)
print('MSE test: ', mse(y_test, predictions_test_3))

### Out of the three models, which would you choose? Why?

I would go with the model 3, which employs the temp and windspeed.

Significance: windspeed is statistically significant (estimate 90.34, t = 7.63, p < 0.001), while holiday is not (p = 0.282).
R²: model 3 is the best one with respect to this metric as it provides the best fit for the data, which can be seen from the value of 0.2141, compared to 0.2070 for model 1 and 0.2072 for model 2.
Test MSE: the lowest value of 14182.22 was achieved for model 3, which was better than 14355.47 and 14349.57 attained for other two models. Thus, model 3 demonstrates the best performance in terms of generalizability.
No overfitting: it can be seen from the comparison of train and test MSE (14096.84 vs. 14182.22).

However, the best available model still explains only 21% of the variability in bikers. There is a need for incorporating more variables, which would account for considerable amount of variance, as can be seen from the analysis. In particular, time variables such as hr and season were found to be non-linear.